In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.cluster import KMeans

In [3]:
# def runpca(df, resultpath, plotpath):
#     df = df.groupby('GENE_ID', as_index=False).mean(numeric_only=True)
#     gene_ids = df.iloc[:, 0].astype('string')
#     df = df.iloc[:, 1:]
#     df.index = gene_ids
#     df = df.T.astype('float64')
#     df.columns = df.columns.astype('string')

#     selector = VarianceThreshold(threshold=0.01)
#     X_reduced = selector.fit_transform(df)
#     selected_genes = df.columns[selector.get_support()].astype('string')

#     scaler = StandardScaler()
#     X_scaled = scaler.fit_transform(X_reduced)

#     pca = PCA(n_components=0.95)
#     X_pca = pca.fit_transform(X_scaled)

#     explained_variance_ratio = pca.explained_variance_ratio_
#     cumulative_variance = np.cumsum(explained_variance_ratio)

#     importance = np.abs(pca.components_).sum(axis=0)
#     feature_importance = pd.Series(importance, index=selected_genes)
#     top_features = feature_importance.sort_values(ascending=False).head(50)
#     top_features_df = top_features.reset_index(name='Score').rename(columns={'index':'GENE_ID'})

#     original_df = df.T.reset_index().rename(columns={'index':'GENE_ID'})
#     original_df['GENE_ID'] = original_df['GENE_ID'].astype('string')

#     final_df = pd.merge(
#         original_df, 
#         top_features_df, 
#         on='GENE_ID', 
#         how='inner', 
#         validate='one_to_one'
#     )
#     final_df = final_df.sort_values('Score', ascending=False).reset_index(drop=True)
#     final_df.to_csv(resultpath, index=False)

#     plt.figure(figsize=(10, 5))
#     plt.plot(range(1, len(explained_variance_ratio) + 1), cumulative_variance, marker='o', linestyle='--')
#     plt.xlabel("Number of Principal Components")
#     plt.ylabel("Cumulative Explained Variance")
#     plt.title("PCA Scree Plot (After Feature Selection)")
#     plt.grid()
#     plt.savefig(plotpath, dpi=300, bbox_inches='tight')
#     plt.close()

#     return final_df

In [4]:
# def runpca_cluster(df, resultpath, plotpath, clusterplotpath, n_clusters=3):
#     df = df.groupby('GENE_ID', as_index=False).mean(numeric_only=True)
#     gene_ids = df.iloc[:, 0].astype('string')
#     df = df.iloc[:, 1:]
#     df.index = gene_ids
#     df = df.T.astype('float64')
#     df.columns = df.columns.astype('string')

#     selector = VarianceThreshold(threshold=0.01)
#     X_reduced = selector.fit_transform(df)
#     selected_genes = df.columns[selector.get_support()].astype('string')

#     scaler = StandardScaler()
#     X_scaled = scaler.fit_transform(X_reduced)

#     pca = PCA(n_components=0.95)
#     X_pca = pca.fit_transform(X_scaled)

#     explained_variance_ratio = pca.explained_variance_ratio_
#     cumulative_variance = np.cumsum(explained_variance_ratio)

#     importance = np.abs(pca.components_).sum(axis=0)
#     feature_importance = pd.Series(importance, index=selected_genes)
#     top_features = feature_importance.sort_values(ascending=False).head(50)
#     top_features_df = top_features.reset_index(name='Score').rename(columns={'index':'GENE_ID'})

#     original_df = df.T.reset_index().rename(columns={'index':'GENE_ID'})
#     original_df['GENE_ID'] = original_df['GENE_ID'].astype('string')

#     final_df = pd.merge(
#         original_df, 
#         top_features_df, 
#         on='GENE_ID', 
#         how='inner', 
#         validate='one_to_one'
#     )
#     final_df = final_df.sort_values('Score', ascending=False).reset_index(drop=True)
#     final_df.to_csv(resultpath, index=False)

#     plt.figure(figsize=(10, 5))
#     plt.plot(range(1, len(explained_variance_ratio) + 1), cumulative_variance, marker='o', linestyle='--')
#     plt.xlabel("Number of Principal Components")
#     plt.ylabel("Cumulative Explained Variance")
#     plt.title("PCA Scree Plot (After Feature Selection)")
#     plt.grid()
#     plt.savefig(plotpath, dpi=300, bbox_inches='tight')
#     plt.close()

#     # KMeans clustering in PCA space
#     kmeans = KMeans(n_clusters=n_clusters, random_state=42)
#     clusters = kmeans.fit_predict(X_pca)

#     # Cluster plot in first two PCs
#     plt.figure(figsize=(10, 7))
#     sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=clusters, palette='viridis', legend='full')
#     plt.xlabel('PC1')
#     plt.ylabel('PC2')
#     plt.title(f'PCA Clusters (k={n_clusters})')
#     plt.legend(title='Cluster')
#     plt.grid(True)
#     plt.savefig(clusterplotpath, dpi=300, bbox_inches='tight')
#     plt.close()

#     return final_df, clusters

In [5]:
def runpca_cluster(
    df, 
    resultpath, 
    screeplotpath, 
    geneclusterplotpath, 
    n_clusters=3, 
    n_gene_clusters=5
):
    # --- 1. Preprocessing ---
    df = df.groupby('GENE_ID', as_index=False).mean(numeric_only=True)
    gene_ids = df.iloc[:, 0].astype('string')
    df = df.iloc[:, 1:]
    df.index = gene_ids
    # Now: rows=GENES, columns=SAMPLES

    # --- 2. Sample-centric PCA & Clustering ---
    df_samples = df.T.astype('float64')
    df_samples.columns = df_samples.columns.astype('string')
    
    selector = VarianceThreshold(threshold=0.01)
    X_reduced = selector.fit_transform(df_samples)
    selected_genes = df_samples.columns[selector.get_support()].astype('string')

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_reduced)

    pca = PCA(n_components=0.95)
    X_pca = pca.fit_transform(X_scaled)
    explained_variance_ratio = pca.explained_variance_ratio_
    cumulative_variance = np.cumsum(explained_variance_ratio)

    importance = np.abs(pca.components_).sum(axis=0)
    feature_importance = pd.Series(importance, index=selected_genes)
    top_features = feature_importance.sort_values(ascending=False).head(50)
    top_features_df = top_features.reset_index(name='Score').rename(columns={'index':'GENE_ID'})

    original_df = df_samples.T.reset_index().rename(columns={'index':'GENE_ID'})
    original_df['GENE_ID'] = original_df['GENE_ID'].astype('string')
    final_df = pd.merge(
        original_df, 
        top_features_df, 
        on='GENE_ID', 
        how='inner', 
        validate='one_to_one'
    )
    final_df = final_df.sort_values('Score', ascending=False).reset_index(drop=True)
    final_df.to_csv(resultpath, index=False)  # Only save top genes CSV

    # --- 3. Scree Plot ---
    plt.figure(figsize=(10, 5))
    plt.plot(range(1, len(explained_variance_ratio) + 1), cumulative_variance, marker='o', linestyle='--')
    plt.xlabel("Number of Principal Components")
    plt.ylabel("Cumulative Explained Variance")
    plt.title("PCA Scree Plot (After Feature Selection)")
    plt.grid()
    plt.savefig(screeplotpath, dpi=300, bbox_inches='tight')
    plt.close()

    # --- 4. Sample Clustering (no plot) ---
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    clusters = kmeans.fit_predict(X_pca)

    # --- 5. Gene-centric PCA & Clustering ---
    gene_centered = df.astype('float64')  # rows=GENES, columns=SAMPLES

    scaler_genes = StandardScaler()
    X_genes_scaled = scaler_genes.fit_transform(gene_centered)

    pca_genes = PCA(n_components=2)
    X_pca_genes = pca_genes.fit_transform(X_genes_scaled)

    kmeans_genes = KMeans(n_clusters=n_gene_clusters, random_state=42)
    gene_clusters = kmeans_genes.fit_predict(X_pca_genes)

    # No gene cluster CSV is saved

    plt.figure(figsize=(12, 8))
    ax = sns.scatterplot(x=X_pca_genes[:, 0], y=X_pca_genes[:, 1], 
                         hue=gene_clusters, palette='tab10', legend='full')
    # Only label top 20 genes by variance
    top_var_genes = gene_centered.var(axis=1).sort_values(ascending=False).head(20).index
    for i, gene_id in enumerate(gene_centered.index):
        if gene_id in top_var_genes:
            ax.annotate(gene_id, (X_pca_genes[i, 0], X_pca_genes[i, 1]), 
                        xytext=(3, 3), textcoords='offset points',
                        fontsize=7, alpha=0.8)
    plt.xlabel('Gene PC1')
    plt.ylabel('Gene PC2')
    plt.title(f'Gene PCA Clusters (k={n_gene_clusters})')
    plt.legend(title='Gene Cluster')
    plt.grid(alpha=0.3)
    plt.savefig(geneclusterplotpath, dpi=300, bbox_inches='tight')
    plt.close()

    return final_df, gene_clusters


In [356]:
# filepath='Normalized/GSE38265_normalized.csv'
# df = pd.read_csv(filepath)
# df = df.fillna(0)
# print(df.shape)
# resultpath = 'Results/Features/GSE38265_features.csv'
# plotpath = 'Results/Plot/GSE38265_plot.png'
# clusterplotpath = 'Results/ClusterPlot/GSE38265_clusterplot.png'

# df, clusters = runpca_cluster(df, resultpath, plotpath, clusterplotpath, n_clusters=4)

In [357]:
filepath='Normalized/GSE38265.csv'
df = pd.read_csv(filepath)
df = df.fillna(0)
print(df.shape)
resultpath = 'Results/Features/GSE38265new_features.csv'
plotpath = 'Results/Plot/GSE38265new_plot.png'
clusterplotpath = 'Results/ClusterPlot/GSE38265new_clusterplot.png'

df, clusters = runpca_cluster(df, resultpath, plotpath, clusterplotpath, n_clusters=3)

(22340, 7)


C:\Users\ankym\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\sklearn\cluster\_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)
C:\Users\ankym\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\sklearn\cluster\_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)


In [358]:
# filepath='Normalized/GSE61358_normalized.csv'
# df = pd.read_csv(filepath)
# df = df.fillna(0)
# print(df.shape)
# resultpath = 'Results/Features/GSE61358_features.csv'
# plotpath = 'Results/Plot/GSE61358_plot.png'
# clusterplotpath = 'Results/ClusterPlot/GSE61358_clusterplot.png'

# df, clusters = runpca_cluster(df, resultpath, plotpath, clusterplotpath, n_clusters=4)

In [359]:
filepath='Normalized/GSE68559_genenames.csv'
df = pd.read_csv(filepath)
df = df.fillna(0)
print(df.shape)
resultpath = 'Results/Features/GSE68559_features.csv'
plotpath = 'Results/Plot/GSE68559_plot.png'
clusterplotpath = 'Results/ClusterPlot/GSE68559_clusterplot.png'

df, clusters = runpca_cluster(df, resultpath, plotpath, clusterplotpath, n_clusters=4)

(39376, 12)


C:\Users\ankym\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\sklearn\cluster\_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)
C:\Users\ankym\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\sklearn\cluster\_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)


In [360]:
filepath='Normalized/GSE69626_genenames.csv'
df = pd.read_csv(filepath)
df = df.fillna(0)
print(df.shape)
resultpath = 'Results/Features/GSE69626_features.csv'
plotpath = 'Results/Plot/GSE69626_plot.png'
clusterplotpath = 'Results/ClusterPlot/GSE69626_clusterplot.png'

df, clusters = runpca_cluster(df, resultpath, plotpath, clusterplotpath, n_clusters=4)

(39376, 27)


C:\Users\ankym\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\sklearn\cluster\_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)
C:\Users\ankym\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\sklearn\cluster\_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)


In [361]:
# filepath='Normalized/GSE239320_normalized.csv'
# df = pd.read_csv(filepath)
# df = df.fillna(0)
# print(df.shape)
# resultpath = 'Results/Features/GSE239320_features.csv'
# plotpath = 'Results/Plot/GSE239320_plot.png'
# clusterplotpath = 'Results/ClusterPlot/GSE239320_clusterplot.png'

# df, clusters = runpca_cluster(df, resultpath, plotpath, clusterplotpath, n_clusters=4)

In [362]:
filepath='Normalized/GSE239446_genenames.csv'
df = pd.read_csv(filepath)
df = df.fillna(0)
print(df.shape)
resultpath = 'Results/Features/GSE239446_features.csv'
plotpath = 'Results/Plot/GSE239446_plot.png'
clusterplotpath = 'Results/ClusterPlot/GSE239446_clusterplot.png'

df, clusters = runpca_cluster(df, resultpath, plotpath, clusterplotpath, n_clusters=4)

(62707, 25)


C:\Users\ankym\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\sklearn\cluster\_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)
C:\Users\ankym\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\sklearn\cluster\_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)


In [363]:
filepath='Normalized/GSE251814_normalized.csv'
df = pd.read_csv(filepath)
df = df.fillna(0)
print(df.shape)
resultpath = 'Results/Features/GSE251814_features.csv'
plotpath = 'Results/Plot/GSE251814_plot.png'
clusterplotpath = 'Results/ClusterPlot/GSE251814_clusterplot.png'

df, clusters = runpca_cluster(df, resultpath, plotpath, clusterplotpath, n_clusters=4)

(60660, 19)


C:\Users\ankym\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\sklearn\cluster\_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)
C:\Users\ankym\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\sklearn\cluster\_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)


In [364]:
filepath='Normalized/GSE266663_genenames.csv'
df = pd.read_csv(filepath)
df = df.fillna(0)
print(df.shape)
resultpath = 'Results/Features/GSE266663_features.csv'
plotpath = 'Results/Plot/GSE266663_plot.png'
clusterplotpath = 'Results/ClusterPlot/GSE266663_clusterplot.png'

df, clusters = runpca_cluster(df, resultpath, plotpath, clusterplotpath, n_clusters=4)

(39376, 10)


C:\Users\ankym\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\sklearn\cluster\_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)
C:\Users\ankym\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\sklearn\cluster\_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)


In [8]:
filepath='Normalized/GSE273125.csv'
df = pd.read_csv(filepath)
df = df.fillna(0)
print(df.shape)
resultpath = 'Results/Features/GSE273125_features.csv'
plotpath = 'Results/Plot/GSE273125_plot.png'
clusterplotpath = 'Results/ClusterPlot/GSE273125_clusterplot.png'

df, clusters = runpca_cluster(df, resultpath, plotpath, clusterplotpath, n_clusters=4)

(62708, 25)


C:\Users\ankym\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\sklearn\cluster\_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)
C:\Users\ankym\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\sklearn\cluster\_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)


In [366]:
# filepath='Normalized/GSE193571_normalized.csv'
# df = pd.read_csv(filepath)
# df = df.fillna(0)
# print(df.shape)
# resultpath = 'Results/Features/GSE193571_features.csv'
# plotpath = 'Results/Plot/GSE193571_plot.png'
# clusterplotpath = 'Results/ClusterPlot/GSE193571_clusterplot.png'

# df, clusters = runpca_cluster(df, resultpath, plotpath, clusterplotpath, n_clusters=4)

In [7]:
filepath='Normalized/GSE101132.csv'
df = pd.read_csv(filepath)
df = df.fillna(0)
print(df.shape)
resultpath = 'Results/Features/GSE101132_features.csv'
plotpath = 'Results/Plot/GSE101132_plot.png'
clusterplotpath = 'Results/ClusterPlot/GSE101132_clusterplot.png'

df, clusters = runpca_cluster(df, resultpath, plotpath, clusterplotpath, n_clusters=4)

(60499, 46)


C:\Users\ankym\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\sklearn\cluster\_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)
C:\Users\ankym\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\sklearn\cluster\_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)


In [8]:
filepath='Normalized/GSE104406.csv'
df = pd.read_csv(filepath)
df = df.fillna(0)
print(df.shape)
resultpath = 'Results/Features/GSE104406_features.csv'
plotpath = 'Results/Plot/GSE104406_plot.png'
clusterplotpath = 'Results/ClusterPlot/GSE104406_clusterplot.png'

df, clusters = runpca_cluster(df, resultpath, plotpath, clusterplotpath, n_clusters=4)

(57260, 21)


C:\Users\ankym\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\sklearn\cluster\_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)
C:\Users\ankym\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\sklearn\cluster\_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)


In [9]:
filepath='Normalized/GSE113253_AT.csv'
df = pd.read_csv(filepath)
df = df.fillna(0)
print(df.shape)
resultpath = 'Results/Features/GSE113253_AT_features.csv'
plotpath = 'Results/Plot/GSE113253_AT_plot.png'
clusterplotpath = 'Results/ClusterPlot/GSE113253_AT_clusterplot.png'

df, clusters = runpca_cluster(df, resultpath, plotpath, clusterplotpath, n_clusters=4)

(17475, 43)


C:\Users\ankym\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\sklearn\cluster\_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)
C:\Users\ankym\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\sklearn\cluster\_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)


In [10]:
filepath='Normalized/GSE113253_BM.csv'
df = pd.read_csv(filepath)
df = df.fillna(0)
print(df.shape)
resultpath = 'Results/Features/GSE113253_BM_features.csv'
plotpath = 'Results/Plot/GSE113253_BM_plot.png'
clusterplotpath = 'Results/ClusterPlot/GSE113253_BM_clusterplot.png'

df, clusters = runpca_cluster(df, resultpath, plotpath, clusterplotpath, n_clusters=4)

(17475, 54)


C:\Users\ankym\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\sklearn\cluster\_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)
C:\Users\ankym\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\sklearn\cluster\_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)


In [ ]:
filepath='Normalized/GSE132154.csv'
df = pd.read_csv(filepath)
df = df.fillna(0)
print(df.shape)
resultpath = 'Results/Features/GSE132154_features.csv'
plotpath = 'Results/Plot/GSE132154_plot.png'
clusterplotpath = 'Results/ClusterPlot/GSE132154_clusterplot.png'

df, clusters = runpca_cluster(df, resultpath, plotpath, clusterplotpath, n_clusters=4)

(39376, 8)


C:\Users\ankym\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\sklearn\cluster\_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)
C:\Users\ankym\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\sklearn\cluster\_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)


In [12]:
filepath='Normalized/GSE139273.csv'
df = pd.read_csv(filepath)
df = df.fillna(0)
print(df.shape)
resultpath = 'Results/Features/GSE139273_features.csv'
plotpath = 'Results/Plot/GSE139273_plot.png'
clusterplotpath = 'Results/ClusterPlot/GSE139273_clusterplot.png'

df, clusters = runpca_cluster(df, resultpath, plotpath, clusterplotpath, n_clusters=4)

(60448, 14)


C:\Users\ankym\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\sklearn\cluster\_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)
C:\Users\ankym\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\sklearn\cluster\_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)


In [13]:
filepath='Normalized/GSE164425.csv'
df = pd.read_csv(filepath)
df = df.fillna(0)
print(df.shape)
resultpath = 'Results/Features/GSE164425_features.csv'
plotpath = 'Results/Plot/GSE164425_plot.png'
clusterplotpath = 'Results/ClusterPlot/GSE164425_clusterplot.png'

df, clusters = runpca_cluster(df, resultpath, plotpath, clusterplotpath, n_clusters=4)

(38692, 26)


C:\Users\ankym\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\sklearn\cluster\_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)
C:\Users\ankym\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\sklearn\cluster\_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)


In [14]:
filepath='Normalized/GSE252276.csv'
df = pd.read_csv(filepath)
df = df.fillna(0)
print(df.shape)
resultpath = 'Results/Features/GSE252276_features.csv'
plotpath = 'Results/Plot/GSE252276_plot.png'
clusterplotpath = 'Results/ClusterPlot/GSE252276_clusterplot.png'

df, clusters = runpca_cluster(df, resultpath, plotpath, clusterplotpath, n_clusters=4)

(39376, 13)


C:\Users\ankym\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\sklearn\cluster\_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)
C:\Users\ankym\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\sklearn\cluster\_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)
